In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.1 Complex Vector Spaces and Inner Products

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.1",
    title="Complex Vector Spaces and Inner Products",
    blurb="The arena. Quantum mechanics happens in a complex vector space, where "
    "a state is a unit vector, the inner product of two states is a probability "
    "amplitude, and the complex numbers — far from a bookkeeping convenience — "
    "are the source of interference. We build that space carefully and compute "
    "in it, meeting the one subtlety that will recur all volume: a global phase "
    "means nothing, a relative phase means everything.",
    difficulty="intermediate",
    estimate="120–150 min",
)

## Notebook overview

This is the first notebook of Volume VI, and it opens not with quantum physics but with the
*arena* in which all of it takes place: the complex vector space. The strategy is the one Volume V
used to such effect — build the mathematical toolkit first, cleanly and for its own sake, so that
when the physics arrives it is never interrupted to stop and learn a technique. Volume VI's opening
movement is that toolkit, and it begins here, with the single object on which the entire theory is
built: a vector of complex numbers.

Two facts about quantum states force this arena upon us, and both are worth feeling before we
formalize them. The first is that states **superpose**: if a system can be in a state $|a\rangle$
and in a state $|b\rangle$, it can also be in any combination $\alpha|a\rangle+\beta|b\rangle$.
Things that add like that live in a **vector space**. The second, stranger fact is that the
coefficients $\alpha,\beta$ are **complex** — they carry a phase, and phases *interfere*. That
single choice, complex rather than real scalars, is the source of nearly everything that makes
quantum mechanics quantum, and we will see it most sharply at the end of this notebook in the
difference between a global phase (which means nothing) and a relative phase (which means
everything).

We develop the space gently but with full rigour. We will meet the **inner product** $\langle
u|v\rangle$, the machine that turns two states into a complex *amplitude* whose squared magnitude
is a probability; the **norm** and **normalization**, which make a physical state a *unit* vector
and encode the total probability $\sum p=1$ of [§5.2](../05-classical-stat-mech/probability.ipynb); **orthonormal bases** and the **expansion
coefficients** that are the components we measure in; the **Cauchy–Schwarz** and **triangle**
inequalities that give the space its geometry; and the global-versus-relative-phase distinction.
The physics is woven in from the first line — amplitudes, probabilities, interference — but we
stay honest about what we are doing: building the stage, not yet the play.

Throughout, every exercise follows the Volume VI contract: a clear statement that fixes
the givens, the goal, and the notation with no hidden assumptions, followed by explicit
enumerated parts naming the exact methods, so nothing is ever reverse-engineered. The difficulty lives in
the mathematics, never in deciphering the question.

> **A word on notation (and its boundary).** We write a state vector as a **ket** $|\psi\rangle$
> and the inner product of two states as $\langle u|v\rangle$, from the very start — this is the
> universal shorthand of the subject, and you will see it everywhere. For now treat $\langle
> u|v\rangle$ simply as *the inner product of $|u\rangle$ and $|v\rangle$*. The deeper reading of
> the **bra** $\langle u|$ as an object in its own right — a dual vector, a linear functional — and
> the outer products and projectors that come with it, is the formal Dirac machinery of [§6.3](dirac-notation-spectral-decomposition.ipynb); we do
> not need it yet. We also stay strictly **finite-dimensional** here, in $\mathbb{C}^n$. The word
> "Hilbert space" adds one more ingredient, *completeness*, which is automatic in finite dimensions
> and becomes the subtle heart of infinite-dimensional (wavefunction) spaces in [§6.9](position-representation.ipynb).
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the conjugate-first inner product matching `numpy.vdot`; Hermitian symmetry and
> sesquilinearity; a normalized state being a unit vector; expansion coefficients reconstructing a
> state with Parseval's identity; Gram–Schmidt producing an orthonormal basis to machine precision;
> the Cauchy–Schwarz and triangle inequalities; and a global phase leaving every probability
> unchanged while a relative phase sweeps one from 1 to 0. A ✓ is strong evidence; a ✗ is a prompt
> to *locate the discrepancy*.
>
> **Scope.** The finite-dimensional complex inner-product space, computationally. Operators and the
> spectral theorem are [§6.2](operators-spectral-theorem.ipynb); formal Dirac notation (the bra, projectors) is [§6.3](dirac-notation-spectral-decomposition.ipynb); the Born rule and
> the postulates, where $|\langle e_i|\psi\rangle|^2$ becomes a measurement probability, are [§6.5](postulates.ipynb);
> infinite-dimensional spaces and wavefunctions are [§6.9](position-representation.ipynb). See Sakurai & Napolitano, *Modern Quantum
> Mechanics*; Nielsen & Chuang, *Quantum Computation and Quantum Information*; Axler, *Linear
> Algebra Done Right*; and Notebooks [§0.4](../00-foundations/linear-systems.ipynb)–[§0.5](../00-foundations/eigenvalues-svd.ipynb) (linear algebra), [§5.2](../05-classical-stat-mech/probability.ipynb) ($\sum p=1$).

## Theory in brief

### Why complex, and why vectors

A quantum state is a vector in a **complex** vector space. Two facts force this: states
**superpose** (the sum of two states is a state — a vector space), and the scalars are **complex**
(amplitudes carry a phase that interferes). Concretely a state is a column of complex numbers — its
components in some basis,

```{math}
:label: eq-state-vector
|\psi\rangle=\begin{pmatrix}\psi_1\\\vdots\\\psi_n\end{pmatrix}\in\mathbb{C}^n,\qquad
\text{a qubit is }|\psi\rangle=\alpha|0\rangle+\beta|1\rangle\in\mathbb{C}^2 .
```

### The Hermitian inner product

The inner product of $|u\rangle$ and $|v\rangle$ is **conjugate-linear in the first slot and linear
in the second** (the physics convention),

```{math}
:label: eq-inner-product
\langle u|v\rangle=\sum_i u_i^{*}\,v_i,\qquad \langle u|v\rangle=\langle v|u\rangle^{*}\ \text{(Hermitian)} .
```

`numpy.vdot(u, v)` conjugates its **first** argument, so it matches this convention exactly — a
point worth flagging, because the opposite (math) convention conjugates the second, and the mismatch
is a classic bug. The inner product turns two states into a complex **amplitude**; $|\langle
u|v\rangle|^2$ will be a probability (the Born rule, [§6.5](postulates.ipynb)).

### Norm and normalization

The norm is $\|\psi\|=\sqrt{\langle\psi|\psi\rangle}$, and $\langle\psi|\psi\rangle$ is real and
non-negative,

```{math}
:label: eq-norm
\|\psi\|=\sqrt{\langle\psi|\psi\rangle},\qquad \text{a physical state is normalized: }\langle\psi|\psi\rangle=1 .
```

A physical state is a **unit vector**. This is exactly the $\sum p=1$ of [§5.2](../05-classical-stat-mech/probability.ipynb) — the total
probability is one.

### Orthonormal bases and components

An orthonormal set satisfies $\langle e_i|e_j\rangle=\delta_{ij}$. In such a basis any state expands
with **expansion coefficients** given by inner products,

```{math}
:label: eq-orthonormal
|\psi\rangle=\sum_i c_i|e_i\rangle,\quad c_i=\langle e_i|\psi\rangle,\qquad
\|\psi\|^2=\sum_i|c_i|^2\ \text{(Parseval)} .
```

For a normalized state $\sum_i|c_i|^2=1$, so the $|c_i|^2$ are probabilities — the probabilities of
the outcomes labelled by the basis (previewed; the full story is [§6.5](postulates.ipynb)).

### Cauchy–Schwarz and the triangle inequality

Two inequalities give the space its geometry,

```{math}
:label: eq-cauchy-schwarz
|\langle u|v\rangle|\le\|u\|\,\|v\|\quad(\text{equality iff }|u\rangle\parallel|v\rangle),
```

```{math}
:label: eq-triangle
\|u+v\|\le\|u\|+\|v\| .
```

Cauchy–Schwarz (proved by minimizing $\|u-\lambda v\|^2$) guarantees the overlap of two normalized
states has magnitude $\le1$ — a probability amplitude is bounded — and it is the seed of the
uncertainty relation ([§6.6](pauli-uncertainty.ipynb)); the triangle inequality follows from it.

### Global versus relative phase — the deep point

Multiplying a *whole* state by a phase changes **no** probability,

```{math}
:label: eq-phase
|\psi\rangle\to e^{i\theta}|\psi\rangle:\quad |\langle\phi|e^{i\theta}\psi\rangle|^2=|\langle\phi|\psi\rangle|^2 ,
```

so a **global phase is physically meaningless** — the true physical states are **rays** (states up
to a global phase), not vectors, and the state space is *projective*. But the **relative** phase
*within* a superposition, $|0\rangle+e^{i\alpha}|1\rangle$, is fully physical: it is what
interferes. This distinction — global phase nothing, relative phase everything — recurs all volume;
for a qubit it is the difference between the global phase the Bloch sphere quotients away and the position on it ([§6.8](bloch-sphere-entanglement.ipynb)).

### Hilbert space

A **Hilbert space** is a complete inner-product space {eq}`eq-hilbert`. Completeness (every Cauchy
sequence converges) is automatic in finite dimensions and is the subtle ingredient that makes
infinite-dimensional function spaces work ([§6.9](position-representation.ipynb)). For this notebook, "Hilbert space" means
"finite-dimensional complex inner-product space, $\mathbb{C}^n$."

```{math}
:label: eq-hilbert
\text{Hilbert space}=\text{complete inner-product space};\quad \text{completeness is automatic in }\mathbb{C}^n .
```

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Convention: the inner product ⟨u|v⟩ = Σ_i conj(u_i) v_i conjugates the FIRST slot
# (the physics convention). numpy.vdot(u, v) conjugates its first argument, so it matches exactly.
# We work in finite-dimensional ℂⁿ throughout.


def inner(u, v):
    """The Hermitian inner product $\\langle u|v\\rangle=\\sum_i u_i^{*}v_i$ {eq}`eq-inner-product`.

    Conjugate-linear in the first slot, linear in the second (the physics convention). This wraps
    ``numpy.vdot(u, v)``, which conjugates its **first** argument and so matches the convention — the
    opposite (mathematics) convention conjugates the second, and confusing the two is a classic bug.

    Parameters
    ----------
    u, v : numpy.ndarray
        Complex vectors of equal length.

    Returns
    -------
    complex
        The inner product $\\langle u|v\\rangle$.
    """
    return np.vdot(u, v)


def normalize(psi):
    """Return the unit vector $|\\psi\\rangle/\\|\\psi\\|$ in the direction of ``psi`` {eq}`eq-norm`.

    A physical quantum state is a unit vector, $\\langle\\psi|\\psi\\rangle=1$ — the statistical-
    mechanics $\\sum p=1$ of §5.2. Dividing by the norm $\\|\\psi\\|=\\sqrt{\\langle\\psi|\\psi\\rangle}$
    (``numpy.linalg.norm``) produces it.

    Parameters
    ----------
    psi : numpy.ndarray
        A nonzero complex vector.

    Returns
    -------
    numpy.ndarray
        The normalized vector.
    """
    return psi / np.linalg.norm(psi)


def gram_schmidt(vectors):
    """Orthonormalize a list of complex vectors by the Gram–Schmidt procedure {eq}`eq-orthonormal`.

    For each vector in turn, subtract its projection onto every previously-accepted orthonormal
    vector (the projection of $|w\\rangle$ onto a unit $|q\\rangle$ is $\\langle q|w\\rangle|q\\rangle$),
    then normalize the remainder. The result is an orthonormal set spanning the same subspace, with
    Gram matrix equal to the identity to machine precision.

    Parameters
    ----------
    vectors : sequence of numpy.ndarray
        Linearly independent complex vectors.

    Returns
    -------
    numpy.ndarray, shape (k, n)
        The orthonormal vectors, one per row.
    """
    basis = []
    for w in vectors:
        w = np.asarray(w, dtype=complex).copy()
        for q in basis:
            w = (
                w - np.vdot(q, w) * q
            )  # subtract the projection onto each previous basis vector
        basis.append(w / np.linalg.norm(w))
    return np.array(basis)

## Exercise 1 — Complex vectors and the inner product

Let $|u\rangle=(1+i,\ 2,\ -i,\ 3-i)^{\mathsf T}$ and $|v\rangle=(2,\ 1-i,\ i,\ 1)^{ \mathsf T}$ be
vectors in $\mathbb{C}^4$. Compute the inner product $\langle u|v\rangle=\sum_i u_i^{*}v_i$, and
verify its three defining properties: that `numpy.vdot` reproduces it (conjugating the *first*
argument), that it is Hermitian-symmetric ($\langle u|v\rangle=\langle v|u\rangle^{*}$), and that
it is conjugate-linear in the first slot and linear in the second {eq}`eq-state-vector`,
{eq}`eq-inner-product`.

1. Construct the two complex vectors.
2. Compute $\langle u|v\rangle$ two ways — by hand as $\sum_i u_i^{*}v_i$ with `numpy.conj`, and
   with `numpy.vdot(u, v)` — and confirm they agree, which shows `vdot` conjugates the first
   argument.
3. Verify Hermitian symmetry by comparing $\langle u|v\rangle$ to $\langle v|u\rangle^{*}$.
4. Verify sesquilinearity: conjugate-linearity in the first slot, $\langle\lambda
   u|v\rangle=\lambda^{*}\langle u|v\rangle$, and linearity in the second, $\langle u|\lambda
   v\rangle=\lambda\langle u|v\rangle$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    np.vdot(u, v),
    np.sum(np.conj(u) * v),
    "the inner product ⟨u|v⟩ conjugates the first argument (physics convention; np.vdot matches it)",
    rtol=1e-12,
)
validate.check(
    np.isclose(np.vdot(u, v), np.conj(np.vdot(v, u)))
    and conj_linear_first
    and linear_second,
    "the inner product is Hermitian-symmetric and sesquilinear (conjugate-linear in the first slot, linear in the second)",
)

## Exercise 2 — Norm and normalization

For the vector $|\psi\rangle=(1+i,\ 1-i,\ 2i,\ 1)^{\mathsf T}\in\mathbb{C}^4$, compute its norm
$\|\psi\|=\sqrt{\langle\psi|\psi\rangle}$ and produce the normalized (unit) vector
$|\hat\psi\rangle$. Confirm $\langle\psi|\psi\rangle$ is real and non-negative and that
$\langle\hat\psi|\hat\psi\rangle=1$ {eq}`eq-norm`.

1. Compute $\langle\psi|\psi\rangle$ with `numpy.vdot(psi, psi)` and check it is real (inspect the
   `.imag` attribute) and $\ge0$.
2. Take the norm $\|\psi\|=\sqrt{\langle\psi|\psi \rangle}$ with `numpy.linalg.norm`.
3. Normalize with the `normalize` helper (which divides by `numpy.linalg.norm`),
   $|\hat\psi\rangle=|\psi\rangle/\|\psi\|$.
4. Confirm $\langle\hat\psi|\hat\psi \rangle=1$ again with `numpy.vdot` — a physical state is a
   unit vector, the $\sum p=1$ of [§5.2](../05-classical-stat-mech/probability.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.vdot(psi, psi).imag,
    0.0,
    "⟨ψ|ψ⟩ is real (the overlap of a state with itself)",
    atol=1e-12,
)
validate.close(
    np.vdot(psi_hat, psi_hat).real,
    1.0,
    "a normalized state is a unit vector, ⟨ψ|ψ⟩=1 (the total probability is one)",
    rtol=1e-12,
)

## Exercise 3 — Orthonormal bases and expansion coefficients

The four vectors $|e_0\rangle,\dots,|e_3\rangle$ obtained below form an orthonormal basis of
$\mathbb{C}^4$. For the normalized state $|\hat\psi\rangle$ of Exercise 2, find its components
$c_i=\langle e_i|\hat\psi\rangle$ in this basis, reconstruct the state as $\sum_i c_i
|e_i\rangle$, and verify Parseval's identity $\|\hat\psi\|^2=\sum_i|c_i|^2$ {eq}`eq-orthonormal`.

1. Verify the basis is orthonormal by building its **Gram matrix** $G_{ij}=\langle e_i|e_j\rangle$
   with `numpy.vdot` and checking it equals `numpy.eye(4)`.
2. Compute the components $c_i=\langle e_i|\hat\psi\rangle$ with `numpy.vdot(e_i, psi)` (project
   onto each basis vector).
3. Reconstruct $|\hat\psi\rangle=\sum_i c_i|e_i\rangle$ as a Python sum and confirm it matches the
   original (`numpy.allclose`).
4. Verify Parseval, $\sum_i|c_i|^2=\|\hat\psi\|^2=1$ (`numpy.abs` squared, summed), so the
   $|c_i|^2$ sum to one — they are the probabilities of the outcomes labelled by the basis
   (previewed; the full story is [§6.5](postulates.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    recon_error,
    0.0,
    "the expansion coefficients c_i=⟨e_i|ψ⟩ reconstruct the state",
    atol=1e-12,
)
validate.close(
    np.sum(np.abs(c) ** 2),
    1.0,
    "Parseval's identity: Σ|c_i|² = ‖ψ‖² = 1 (the |c_i|² are probabilities)",
    rtol=1e-12,
)

## Exercise 4 — Building an orthonormal basis: Gram–Schmidt

Given four linearly independent complex vectors in $\mathbb{C}^4$, construct an orthonormal basis
from them by the Gram–Schmidt procedure, and verify the result is orthonormal to machine precision
{eq}`eq-orthonormal`.

1. State the Gram–Schmidt procedure as numbered steps: for each vector in turn, subtract its
   projection onto every previously-accepted orthonormal vector — the projection of $|w\rangle$
   onto a unit $|q\rangle$ is $\langle q|w\rangle\,|q\rangle$ — then normalize what remains.
2. Apply it with the `gram_schmidt` helper (which uses `numpy.vdot` for the projections and
   `numpy.linalg.norm` to normalize).
3. Verify the output is orthonormal by building its Gram matrix with `numpy.vdot` and checking it
   equals `numpy.eye(4)`, $\max_{ij}|\langle e_i|e_j\rangle- \delta_{ij}|\approx0$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    max_off,
    0.0,
    "Gram–Schmidt produces an orthonormal basis (its Gram matrix is the identity)",
    atol=1e-12,
)

## Exercise 5 — The Cauchy–Schwarz inequality

Verify the Cauchy–Schwarz inequality $|\langle u|v\rangle|\le\|u\|\,\|v\|$ for arbitrary complex
vectors, and confirm that equality holds exactly when $|u\rangle$ and $|v\rangle$ are parallel
{eq}`eq-cauchy-schwarz`.

1. Sketch the proof: the vector $|u\rangle-\lambda|v\rangle$ has non-negative norm for every
   $\lambda$; choosing $\lambda=\langle v|u\rangle/\langle v|v\rangle$ (the value that minimizes
   $\|u-\lambda v\|^2$) and expanding $0\le\|u-\lambda v\|^2$ gives $|\langle
   u|v\rangle|^2\le\langle u|u\rangle\langle v|v\rangle$, which is the inequality.
2. Verify it numerically on several random complex pairs, comparing $|\langle u|v\rangle|$
   (`abs(numpy.vdot(u, v))`) against $\|u\|\,\|v\|$ (`numpy.linalg.norm`).
3. Show equality holds when $|v\rangle\parallel |u\rangle$ (build $|v\rangle=\mu|u\rangle$ by
   scalar multiplication). Its importance: it bounds the overlap of two normalized states by $1$
   (a probability amplitude is bounded) and seeds the uncertainty relation ([§6.6](pauli-uncertainty.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    np.all(margins >= -1e-12),
    "the Cauchy–Schwarz inequality |⟨u|v⟩| ≤ ‖u‖‖v‖ holds for all tested pairs",
)
validate.close(
    equality_gap,
    0.0,
    "Cauchy–Schwarz is an equality exactly when the vectors are parallel",
    atol=1e-10,
)

## Exercise 6 — The triangle inequality

Verify the triangle inequality $\|u+v\|\le\|u\|+\|v\|$ for arbitrary complex vectors
{eq}`eq-triangle`.

1. Derive it from Cauchy–Schwarz: expand $\|u+v\|^2=\|u\|^2+2\,\mathrm{Re}\langle
   u|v\rangle+\|v\|^2$, bound $\mathrm{Re}\langle u|v\rangle\le|\langle u|v\rangle|\le\|u\|\|v\|$,
   and recognize the right-hand side as $(\|u\|+\|v\|)^2$.
2. Verify numerically with `numpy.linalg.norm`, checking $\|u+v\|\le\|u\|+\|v\|$ on random pairs.
   This is the geometry of the space — distances obey the same triangle law as in ordinary
   geometry.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(triangle_ok, "the triangle inequality ‖u+v‖ ≤ ‖u‖+‖v‖ holds")

## Exercise 7 — Global versus relative phase

Demonstrate the central phase distinction of quantum mechanics. First, show that multiplying a
state by a **global** phase, $|\psi\rangle\to e^{i\theta}|\psi\rangle$, changes no probability:
$|\langle\phi|e^{i\theta}\psi\rangle|^2=|\langle\phi|\psi\rangle|^2$ for any $|\phi \rangle$.
Then, for the qubit superposition $|\psi(\alpha)\rangle=(|0\rangle+e^{i\alpha}|1\rangle)/ \sqrt2$
and the fixed state $|{+}\rangle=(|0\rangle+|1\rangle)/\sqrt2$, show that the **relative** phase
$\alpha$ *does* change the probability $P(\alpha)=|\langle{+}|\psi(\alpha)\rangle|^2$, sweeping it
from $1$ to $0$ {eq}`eq-phase`.

1. Build a state $|\psi\rangle$ and a test state $|\phi\rangle$ (the `normalize` helper); apply
   the global phase with `numpy.exp(1j*theta)` and compute $|\langle\phi|\psi\rangle|^2$ and
   $|\langle\phi|e^{i\theta}\psi\rangle|^2$ with `abs(numpy.vdot(...))**2` for a representative
   $\theta$, confirming they are equal — the global phase is unphysical.
2. Form the superposition $|\psi(\alpha)\rangle$ (the relative phase $e^{i\alpha}$ via
   `numpy.exp`) and the state $|{+}\rangle$.
3. Compute $P(\alpha)=$`abs(numpy.vdot(plus, psi_alpha))**2` for $\alpha=0,\ \pi/2,\ \pi$ and
   watch it run $1\to\tfrac12\to0$; plot the full fringe over a `numpy.linspace` grid with
   `matplotlib`.
4. Note the consequence: physical states are **rays** (states up to a global phase), not vectors,
   and the relative phase is the seat of interference — for a qubit, the position on the Bloch
   sphere ([§6.8](bloch-sphere-entanglement.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    P_phased,
    P_plain,
    "a global phase leaves every probability unchanged, |⟨φ|e^(iθ)ψ⟩|² = |⟨φ|ψ⟩|² (physical states are rays)",
    rtol=1e-12,
)
validate.close(
    [P_of_alpha[0.0], P_of_alpha[np.pi / 2], P_of_alpha[np.pi]],
    [1.0, 0.5, 0.0],
    "the relative phase within a superposition is physical: P(α) sweeps 1 → ½ → 0",
    atol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Overlap, distinguishability, and orthogonality

For two normalized states $|\phi\rangle$ and $|\psi\rangle$ in $\mathbb{C}^n$, the magnitude of
their overlap $|\langle\phi|\psi\rangle|$ measures how alike they are. Show that orthogonal states
have overlap $0$ and that states equal up to a phase have overlap $1$, and interpret
$|\langle\phi|\psi\rangle|^2$ as the probability of "passing a test for $\phi$" when the system is
prepared in $\psi$ — a preview of the Born rule {eq}`eq-inner-product`, {eq}`eq-cauchy-schwarz`.

1. Build two normalized states with the `normalize` helper, including an orthogonal pair and a
   phase-shifted copy (`numpy.exp(1j*θ)` times a state).
2. Compute $|\langle\phi|\psi \rangle|$ for each with `abs(numpy.vdot(phi, psi))`.
3. Show the extremes: orthogonal $\Rightarrow 0$, equal-up-to-phase $\Rightarrow 1$ (the
   Cauchy–Schwarz bound, attained).
4. Read $|\langle\phi |\psi\rangle|^2$ as the probability that a system prepared in $|\psi\rangle$
   is found in $|\phi \rangle$ — full statement in [§6.5](postulates.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    np.isclose(overlap_orth, 0.0, atol=1e-12)
    and np.isclose(overlap_same, 1.0, atol=1e-12)
    and overlap_generic <= 1 + 1e-12,
    "the overlap measures distinguishability: 0 for orthogonal states, 1 for identical-up-to-phase, bounded by 1 (Cauchy–Schwarz)",
)

## Exercise 9 — The arena of quantum mechanics

We have built the stage. A quantum state is a unit vector in a complex inner-product space — more
precisely a **ray**, since a global phase carries no physics. The **inner product** turns two states
into a complex **amplitude**, and the squared magnitude of that amplitude is a **probability**,
bounded by one through Cauchy–Schwarz and summing to one through normalization, exactly the $\sum
p=1$ of [§5.2](../05-classical-stat-mech/probability.ipynb). An **orthonormal basis** resolves any state into **components** whose squared
magnitudes are the probabilities of the outcomes that basis labels, and **Gram–Schmidt** lets us
build such a basis at will. And the **relative phase** within a superposition — invisible to the
global phase, fully alive within — is the seat of interference. Everything in quantum mechanics will
be linear algebra on this space, and we can compute all of it.

**This exercise (synthesis).** No new computation: the arena is the result. We have not done any
quantum *physics* yet — only built the space it lives in. But every strange thing to come —
interference, uncertainty, entanglement — is already implicit in two facts established here: the
scalars are complex, and only relative phases are real. The next notebook ([§6.2](operators-spectral-theorem.ipynb)) brings the actors
onto this stage: the **operators**, whose Hermitian ones are the observables we measure and whose
unitary ones are the symmetries and the dynamics, together with the spectral theorem that
diagonalizes them.

## Notebook summary

Volume VI opens with its mathematical arena, the finite-dimensional complex inner-product space,
built computationally and with the physics woven in.

- **States are complex vectors** {eq}`eq-state-vector`: superposition makes them a vector space,
  complex scalars make phases interfere; a qubit lives in $\mathbb{C}^2$.
- **The Hermitian inner product** {eq}`eq-inner-product`: $\langle u|v\rangle=\sum_i u_i^{*}v_i$,
  conjugate-linear in the first slot (so `numpy.vdot` matches it), Hermitian-symmetric and
  sesquilinear — the machine that makes amplitudes.
- **Norm and normalization** {eq}`eq-norm`: a physical state is a unit vector, $\langle\psi|\psi
  \rangle=1$ — the $\sum p=1$ of [§5.2](../05-classical-stat-mech/probability.ipynb).
- **Orthonormal bases and Parseval** {eq}`eq-orthonormal`: $c_i=\langle e_i|\psi\rangle$ reconstruct
  the state, $\sum_i|c_i|^2=1$, the $|c_i|^2$ are probabilities; Gram–Schmidt builds the basis to
  machine precision.
- **Cauchy–Schwarz and triangle** {eq}`eq-cauchy-schwarz`, {eq}`eq-triangle`: the overlap of two
  normalized states is bounded by one (a bounded amplitude; the seed of uncertainty, [§6.6](pauli-uncertainty.ipynb)), and the
  space obeys the triangle law.
- **Global versus relative phase** {eq}`eq-phase`: a global phase changes no probability (states are
  rays), but the relative phase within a superposition sweeps $P$ from $1$ to $0$ — the seat of
  interference.

We have not done quantum physics, only built the space it lives in — yet interference, uncertainty,
and entanglement are already implicit in two facts: the scalars are complex, and only relative
phases are physical.

## Outlook

- **Operators and the spectral theorem ([§6.2](operators-spectral-theorem.ipynb)).** The actors on this stage: Hermitian operators
  (observables, with real spectra and orthonormal eigenbases) and unitary operators (symmetries and
  dynamics), and the spectral theorem that diagonalizes them.
- **Dirac notation made formal ([§6.3](dirac-notation-spectral-decomposition.ipynb)).** The bra $\langle u|$ as a dual vector in its own right,
  outer products, projectors, and the resolution of the identity — the machinery this notebook used
  $\langle u|v\rangle$ as shorthand for.
- **The Born rule and the postulates ([§6.5](postulates.ipynb)).** Where $|\langle e_i|\psi\rangle|^2$ becomes, by
  postulate, the probability of a measurement outcome.
- **The uncertainty principle ([§6.6](pauli-uncertainty.ipynb)).** Grown from the Cauchy–Schwarz inequality met here.
- **The Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb)) and infinite-dimensional Hilbert spaces ([§6.9](position-representation.ipynb)).** The qubit's ray as a
  point on a sphere; and wavefunctions, where completeness becomes essential.
- **Cross-reference** [§0.4](../00-foundations/linear-systems.ipynb)–[§0.5](../00-foundations/eigenvalues-svd.ipynb) (linear algebra), [§5.2](../05-classical-stat-mech/probability.ipynb) ($\sum p=1$), and forward to [§6.2](operators-spectral-theorem.ipynb), [§6.3](dirac-notation-spectral-decomposition.ipynb), [§6.5](postulates.ipynb), [§6.6](pauli-uncertainty.ipynb),
  [§6.8](bloch-sphere-entanglement.ipynb), [§6.9](position-representation.ipynb).

In [ ]:
from ecp.style import footer

footer()